## Data analysis

In [1]:
import sys
import os

# Add the parent directory (simcode) to sys.path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))

In [2]:
import pandas as pd
import numpy as np
import json
from ecv_rank import *
from src.solvers.weighted_network import ASE
from src.methods import * 

In [3]:
edges = pd.read_csv('wikipedia/squirrel/musae_squirrel_edges.csv', header=None)

In [4]:
edges.columns = edges.iloc[0, :]
edges = edges.drop(edges.index[0])

In [5]:
edges['id1'] = edges['id1'].astype(int)
edges['id2'] = edges['id2'].astype(int)

In [6]:
adj_matrix = np.zeros((edges['id1'].max() + 1, edges['id2'].max() + 1))
adj_matrix[edges['id1'], edges['id2']] = 1
adj_matrix[edges['id2'], edges['id1']] = 1  # Assuming undirected graph
adj_matrix[np.diag_indices_from(adj_matrix)] = 0  # Remove self-loops

In [7]:
with open('wikipedia/squirrel/musae_squirrel_features.json', 'r') as f:
    covariates = json.load(f)

In [8]:
m = -1
for x in covariates:
    for y in covariates[x]:
        if y > m:
            m = y

In [9]:
adj_matrix.shape

cov_matrix = np.zeros(shape=(adj_matrix.shape[0], m+1))

for x in covariates:
    for y in covariates[x]:
        cov_matrix[int(x), int(y)] = y

In [10]:
result = ecv_rank(
    adj_matrix,
    max_k=50,
    B=3,
    holdout_p=0.05,
    weighted=False,
    mode="undirected",
    random_state=1,
)

print("SSE selected rank:", result.sse_rank)
print("AUC selected rank:", result.auc_rank)

SSE selected rank: 31
AUC selected rank: 31


In [ ]:
test = MultivariateACTest(k=55, solver=ASE)
data = {
    'A': adj_matrix,
    'Y': cov_matrix
}

test.fit(data, M=1, n_permutations=1000, random_state=1, 
         permutation_type='covariates', alpha=0.05, verbose=True)